# MNE-bids

In [ ]:
from pathlib import Path
import numpy as np

import openneuro
import mne
import mne_qt_browser

from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids,
)

import matplotlib.pyplot as plt
plt.switch_backend('Qt5Agg')
plt.ion()

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'code'))
from pop_rejcont import rejcont
from preprocessing import preprocess
from psd_lof import find_bad_channels_psd_lof

In [ ]:
dataset = "ds003969"
tasks = ['think1', 'think2','med1breath', 'med2']

# Create a BIDS root directory in the MNE dataset folder
root = Path('/home/jackhoyle')
bids_root = root / 'mne_data' / dataset
bids_root.mkdir(parents=True, exist_ok=True)

In [ ]:
subject = "007"
task = 'med2'

In [ ]:
raw: mne.io.edf.edf.RawEDF = read_raw_bids(
    BIDSPath(subject=subject, root=bids_root, task=task, datatype="eeg"),
    verbose=False
)
raw.load_data()
raw.set_montage("colin27_1020", on_missing="warn")
assert type(raw) == mne.io.edf.edf.RawBDF

In [ ]:
preprocess(
    raw,
    reference='EXG6',
    subsample_freq=256,
    high_thresh=None,
    low_thresh=1,
    pop_rejcont=True,
    bad_channel_variance_threshold=None
)

# outlier/bad channel detection strategies

In [ ]:
psd = raw.compute_psd()

In [ ]:
# built in raw waveform based outlier detection
mne.preprocessing.find_bad_channels_lof(raw, threshold=1.5)

In [ ]:
# power spectrum based outlier detection
find_bad_channels_psd_lof(raw, threshold=2)

In [ ]:
find_bad_channels_psd_lof(raw, threshold=2, log_bands=True)

In [ ]:
psd.plot(picks="eeg", average=False)

In [ ]:
raw.plot()